# AI Code Auditor — Live Demo

**How to run:**
1. Set GPU to T4 x1
2. Run Cell 1 (install) — ~1 min
3. Run Cell 2 (load model) — ~2 min
4. Run Cell 3 (launch interface) — opens a public link
5. Share the public URL with faculty

In [ ]:
# ── Cell 1: Install ────────────────────────────────────────────────────────
!pip install -q \
    transformers==4.40.2 \
    peft==0.10.0 \
    accelerate==0.29.3 \
    bitsandbytes==0.45.3 \
    gradio
print('Done')

In [ ]:
# ── Cell 2: Load model ─────────────────────────────────────────────────────
import os, torch, re
os.environ['LD_LIBRARY_PATH'] = '/usr/local/cuda/lib64:' + os.environ.get('LD_LIBRARY_PATH', '')

from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

# Find adapter
ADAPTER_PATH = None
for root, dirs, files in os.walk('/kaggle/input'):
    for f in files:
        if f == 'adapter_config.json':
            ADAPTER_PATH = root

assert ADAPTER_PATH, 'adapter_config.json not found — add lora adapter dataset'
print(f'Adapter found: {ADAPTER_PATH}')
print(f'CUDA: {torch.cuda.is_available()} | {torch.cuda.get_device_name(0)}')

BASE_MODEL = 'codellama/CodeLlama-7b-hf'

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'left'

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map='auto',
    trust_remote_code=True,
    torch_dtype=torch.float16,
)
base_model.config.use_cache = True

model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)
model.eval()

print('Model loaded and ready!')
print(f'VRAM used: {torch.cuda.memory_allocated()/1e9:.1f} GB')

In [ ]:
# ── Cell 3: Launch Gradio Interface ───────────────────────────────────────
import gradio as gr
import torch, re

def analyze_code(code):
    if not code or len(code.strip()) < 10:
        return 'Please paste some code to analyze.', '', '', ''

    prompt = (
        '<s>[INST] <<SYS>>\n'
        'You are an expert security code auditor.\n'
        '<</SYS>>\n\n'
        'Analyze the following C/C++ code for security vulnerabilities '
        'and provide a secure rewrite:\n\n'
        '```c\n' + code.strip() + '\n``` [/INST]'
    )

    inputs = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=512).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=300,
            do_sample=True,
            temperature=0.2,
            top_p=0.9,
            repetition_penalty=1.3,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.eos_token_id,
        )

    raw = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)

    # Cut off repetitions
    cutoff = raw.find('### Changes Made', raw.find('### Changes Made') + 1)
    if cutoff > 0:
        raw = raw[:cutoff].strip()

    # Extract fields
    cwe_match     = re.search(r'\*\*Vulnerability:\*\*\s*(CWE-\d+)', raw)
    cve_match     = re.search(r'\*\*CVE Reference:\*\*\s*(CVE-[\d-]+)', raw)
    cvss_match    = re.search(r'\*\*CVSS Score:\*\*\s*([\d.]+\s*\(\w+\))', raw)
    code_blocks   = re.findall(r'```(?:c|cpp)?\n(.*?)```', raw, re.DOTALL)
    wrong_match   = re.search(r"### What's Wrong\n(.*?)(?=###|$)", raw, re.DOTALL)
    changes_match = re.search(r'### Changes Made\n(.*?)(?=###|$)', raw, re.DOTALL)

    cwe     = cwe_match.group(1)      if cwe_match     else 'Unknown'
    cve     = cve_match.group(1)      if cve_match     else 'N/A'
    cvss    = cvss_match.group(1)     if cvss_match    else 'N/A'
    secure  = code_blocks[-1].strip() if code_blocks   else 'Could not extract secure rewrite.'
    explain = wrong_match.group(1).strip()   if wrong_match   else ''
    changes = changes_match.group(1).strip() if changes_match else ''

    severity_emoji = '🔴' if 'HIGH' in cvss.upper() or 'CRITICAL' in cvss.upper() else '🟡'

    summary = (
        severity_emoji + ' **' + cwe + '** detected\n\n'
        '📋 **CVE:** ' + cve + '\n'
        '⚠️ **CVSS Score:** ' + cvss + '\n\n'
        "**What's wrong:**\n" + explain
    )

    return summary, secure, changes, raw


EXAMPLES = [
    ['void copy_input(char *user_input) {\n    char buffer[128];\n    strcpy(buffer, user_input);\n    printf("Input: %s\\n", buffer);\n}'],
    ['int read_data(int fd) {\n    char buf[256];\n    int n = read(fd, buf, 1024);\n    buf[n] = 0;\n    return n;\n}'],
    ['void process(int *arr, int size) {\n    int total = 0;\n    for (int i = 0; i <= size; i++) {\n        total += arr[i];\n    }\n}'],
]


with gr.Blocks(title='AI Code Auditor') as demo:

    gr.Markdown(
        '# 🔐 AI Code Auditor\n'
        '### Security Vulnerability Detection using Fine-tuned CodeLlama-7B (QLoRA)\n'
        'Paste any C/C++ code below and click **Analyze** to detect vulnerabilities and get a secure rewrite.'
    )

    with gr.Row():
        with gr.Column(scale=1):
            code_input = gr.Code(
                label='Paste Vulnerable Code Here',
                language='c',
                lines=15,
            )
            analyze_btn = gr.Button('🔍 Analyze Code', variant='primary', size='lg')
            gr.Examples(examples=EXAMPLES, inputs=code_input, label='Example Vulnerable Snippets')

        with gr.Column(scale=1):
            summary_out = gr.Markdown(label='Vulnerability Summary')
            secure_out  = gr.Code(
                label='Secure Rewrite',
                language='c',
                lines=15,
                interactive=False,
            )

    with gr.Accordion('Changes Made', open=False):
        changes_out = gr.Markdown()

    with gr.Accordion('Full Raw Output', open=False):
        raw_out = gr.Markdown()

    analyze_btn.click(
        fn=analyze_code,
        inputs=code_input,
        outputs=[summary_out, secure_out, changes_out, raw_out]
    )

    gr.Markdown(
        '---\n'
        '**Model:** CodeLlama-7B fine-tuned with QLoRA on Big-Vul (3,162 samples)  \n'
        '**Method:** PEFT / QLoRA (4-bit NF4, LoRA r=16)  \n'
        '**Dataset:** Big-Vul — real-world CVE-linked C/C++ vulnerabilities'
    )

demo.launch(share=True)
print('Done! Share the public URL above with your faculty.')